# M2: DNS-BERT + Isolation Forest on CIC (Colab)
Owner: Thierry · Backup: Yehan. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**Put these 5 CIC zips in Google Drive → `MyDrive/leakmate/raw/`** (exact names):
`Benign_1.zip`, `lightBenign.zip`, `lightAttacks.zip`, `heavyBenign.zip`, `heavyAttacks.zip`

The notebook unzips them, builds the query table, trains, and saves results + checkpoints to Drive.
A disconnect resumes from the last checkpoint: just Run all again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/leakmate'
BRANCH = 'claude/quirky-dijkstra-kgq1zg'   # change to main once merged
!mkdir -p $DRIVE/checkpoints $DRIVE/results

In [ ]:
%cd /content
!rm -rf leakmate && git clone -q -b $BRANCH https://github.com/TerminallyStoned/leakmate.git
%cd /content/leakmate
!pip -q install -r requirements-m2.txt

In [ ]:
# Tranco top-1M (benign names, ref [18]); cached on Drive
import os
if not os.path.exists(f'{DRIVE}/tranco.csv'):
    !wget -q -O /tmp/tranco.zip https://tranco-list.eu/top-1m.csv.zip && unzip -p /tmp/tranco.zip > $DRIVE/tranco.csv
!mkdir -p data/external && cp $DRIVE/tranco.csv data/external/tranco.csv && head -3 data/external/tranco.csv

In [ ]:
# Build the CIC query table from the zips on Drive (cached on Drive after the first run)
import os, glob, zipfile
CIC = f'{DRIVE}/data/processed/cic/queries.parquet'
SUB = {'Benign_1': 'benign_main', 'lightBenign': 'light_benign', 'lightAttacks': 'light_attacks',
       'heavyBenign': 'heavy_benign', 'heavyAttacks': 'heavy_attacks'}
if not os.path.exists(CIC):
    for z in glob.glob(f'{DRIVE}/raw/*.zip'):
        name = os.path.basename(z)[:-4]
        dest = f'data/raw/cic/{SUB.get(name, name)}'
        zipfile.ZipFile(z).extractall(dest)
        for inner in glob.glob(f'{dest}/*.zip'):   # heavyAttacks may contain a nested zip
            zipfile.ZipFile(inner).extractall(dest)
    !find data/raw/cic -name '*.pcap' | sort
    !python scripts/cic_to_queries.py --raw data/raw/cic --out $CIC
DATA, EXP = CIC, 'E2'
import pandas as pd
df = pd.read_parquet(DATA); print(df.shape); print(df.groupby(['split', 'label']).size())

## 1. DNS-BERT, BPE tokenizer (main model)

In [ ]:
!python -m models.m2_run --model m2 --exp $EXP --data $DATA \
    --tranco data/external/tranco.csv --tranco-n 200000 \
    --tokenizer bpe --epochs 3 --batch-size 512 \
    --ckpt-dir $DRIVE/checkpoints/m2_bpe --save-model models/weights/m2_bpe

## 2. Ablation 1: character tokenizer

In [ ]:
!python -m models.m2_run --model m2 --exp $EXP --tag ablation1_char --data $DATA \
    --tranco data/external/tranco.csv --tranco-n 200000 \
    --tokenizer char --epochs 3 --batch-size 512 \
    --ckpt-dir $DRIVE/checkpoints/m2_char --save-model models/weights/m2_char

## 3. Isolation Forest baseline (benign-only)

In [ ]:
!python -m models.m2_run --model iforest --exp $EXP --data $DATA --save-model models/weights/iforest.joblib

## 4. Quick look (official metrics come from the shared eval module)

In [ ]:
import json, glob, numpy as np
from sklearn.metrics import roc_auc_score
for f in sorted(glob.glob('results/*/*.json')):
    r = json.load(open(f)); y, s = np.array(r['labels']), np.array(r['scores'])
    auc = roc_auc_score(y, s) if 0 < y.sum() < len(y) else float('nan')
    print(f"{r['model']:22s} {r['experiment']:6s} AUC={auc:.4f} train={r['train_time_s']:.0f}s lat={r['latency_ms_per_query']:.2f}ms")

In [ ]:
# copy results + weights to Drive; push results/ to your branch from your laptop
!cp -r results/* $DRIVE/results/ && cp -r models/weights $DRIVE/ && ls -R $DRIVE/results | head